# 01. EDA — OCT5k 탐색적 데이터 분석

## Google Colab 환경 설정

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    !git clone https://github.com/Lunarian0928/youth_bio_global_portfolio.git
    %cd youth_bio_global_portfolio/oct_retinal_segmentation

    !pip install -q segmentation-models-pytorch albumentations koreanize-matplotlib


## Imports

In [ ]:
import os
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import koreanize_matplotlib
import numpy as np
from PIL import Image

from src.dataset import DEFAULT_DISEASES, OCTRetinalDataset


## Config

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/data/OCT5k" if IN_COLAB else "../data/OCT5k"
GRADING = 1


## 데이터셋 로드

In [ ]:
dataset = OCTRetinalDataset(root_dir=DATA_ROOT, grading=GRADING)
print("총 샘플 수:", len(dataset))


## 질환별 분포 (AMD Part1 / AMD Part2 / DME / Normal Part1 / Normal Part2)

In [ ]:
disease_counts = {disease: 0 for disease in DEFAULT_DISEASES}
for image_path, _ in dataset.samples:
    for disease in DEFAULT_DISEASES:
        if f"{os.sep}{disease}{os.sep}" in image_path:
            disease_counts[disease] += 1
            break

plt.figure(figsize=(8, 4))
plt.bar(disease_counts.keys(), disease_counts.values())
plt.ylabel("샘플 수")
plt.xticks(rotation=20)
plt.title("질환별 샘플 분포 (Grading 1)")
plt.tight_layout()
plt.show()

disease_counts


## 샘플 이미지 시각화

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, idx in enumerate(np.random.choice(len(dataset), size=4, replace=False)):
    image_path, mask_path = dataset.samples[idx]
    image = np.array(Image.open(image_path))
    mask = np.array(Image.open(mask_path))

    axes[0, col].imshow(image, cmap="gray")
    axes[0, col].set_title("Image", fontsize=10)
    axes[0, col].axis("off")

    axes[1, col].imshow(mask, cmap="viridis", vmin=0, vmax=5)
    axes[1, col].set_title("Mask (layer label)", fontsize=10)
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()


## 이미지 크기 및 픽셀 통계

In [ ]:
sizes = set()
mask_value_counts = np.zeros(6, dtype=np.int64)

for image_path, mask_path in dataset.samples:
    image = np.array(Image.open(image_path))
    mask = np.array(Image.open(mask_path))
    sizes.add(image.shape)
    values, counts = np.unique(mask, return_counts=True)
    for v, c in zip(values, counts):
        mask_value_counts[v] += c

print("이미지 크기 종류:", sizes)
print("마스크 클래스별 픽셀 수:", dict(enumerate(mask_value_counts)))
print("배경(0) 비율:", mask_value_counts[0] / mask_value_counts.sum())
